# 05 — CpG Suppression Quantification
**Project:** SARS-CoV-2 Genomic Signatures — Explainable Analysis  
**Author:** Ibrahim Mustafa (Bembo)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/your_file.csv', sep='\t')
df['Clade'] = df['Clade'].str.replace('.', '', regex=False).str.strip()
X = df.drop(columns=['ID', 'Clade'])
y = df['Clade']

colors = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4','#f032e6']
print('Data loaded!')

In [ ]:
# CpG Suppression per Clade
cpg_data = df.groupby('Clade')['CG'].mean().reset_index()
cpg_data.columns = ['Clade', 'Mean_CG']
cpg_data['Clade_short'] = cpg_data['Clade'].str.replace('Clade_', '')
cpg_data = cpg_data.sort_values('Mean_CG')

plt.figure(figsize=(10, 5))
bars = plt.bar(cpg_data['Clade_short'], cpg_data['Mean_CG'], color=colors)
plt.axhline(y=1.0, color='black', linestyle='--', linewidth=1.5, label='Expected (no bias)')
plt.xlabel('Clade')
plt.ylabel('Mean CG Dinucleotide Frequency')
plt.title('CpG Dinucleotide Frequency per Clade\n(values < 1.0 indicate CpG suppression)')
plt.legend()
plt.tight_layout()
plt.savefig('cpg_quantification.png', dpi=150)
plt.show()

print(cpg_data[['Clade_short', 'Mean_CG']].to_string(index=False))
print(f'\nAll clades show ~{((1 - cpg_data["Mean_CG"].mean())*100):.0f}% CpG suppression below expected')

In [ ]:
# Heatmap — Top 20 features per clade
top_features = ['CGG','CCG','GGG','CCC','ATA','CCT','AGG','CAG','CG',
                'TAT','GC','CTG','GCG','CGC','GG','CAC','GTG','GCT','CCA','TGG']

clade_means = df.groupby('Clade')[X.columns.tolist()].mean()

plt.figure(figsize=(14, 6))
sns.heatmap(clade_means[top_features],
            cmap='RdBu_r', center=1.0,
            linewidths=0.5,
            yticklabels=True)
plt.title('Mean k-mer Frequencies per Clade (Top 20 Features)')
plt.xlabel('k-mer')
plt.ylabel('Clade')
plt.tight_layout()
plt.savefig('heatmap_clades.png', dpi=150)
plt.show()